# Bayesian Inference

Companion notebook for the [Bayesian Inference](https://ml-viz.vercel.app/courses/probability-statistics/03-bayesian-inference) lesson.

We'll visualize posterior updating with conjugate priors and show the MAP-regularization connection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#30344a', 'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0', 'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})

## Sequential Bayesian updating — Beta-Bernoulli

We flip a biased coin and update our belief about p after each observation.

In [ ]:
rng = np.random.default_rng(42)
true_p = 0.7
n_obs  = 50
data   = rng.binomial(1, true_p, n_obs)

# Prior: Beta(2, 2) — mild belief coin is fair
alpha0, beta0 = 2, 2

p_range = np.linspace(0.001, 0.999, 500)
snapshots = [0, 1, 5, 20, 50]

fig, axes = plt.subplots(1, len(snapshots), figsize=(18, 4), sharey=False)

cumulative_heads = 0
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(snapshots)))

for i, n in enumerate(snapshots):
    n_heads = data[:n].sum() if n > 0 else 0
    n_tails = n - n_heads
    alpha_post = alpha0 + n_heads
    beta_post  = beta0  + n_tails

    pdf = stats.beta.pdf(p_range, alpha_post, beta_post)
    post_mean = alpha_post / (alpha_post + beta_post)

    axes[i].plot(p_range, pdf, color=colors[i], lw=2)
    axes[i].fill_between(p_range, pdf, alpha=0.2, color=colors[i])
    axes[i].axvline(true_p, color='#2dd4bf', lw=1.5, linestyle=':', label='True p')
    axes[i].axvline(post_mean, color='#f97316', lw=1.5, linestyle='--', label=f'Post. mean={post_mean:.2f}')
    axes[i].set_title(f'After {n} obs\n(H={n_heads}, T={n_tails})', fontsize=9)
    axes[i].set_xlabel('p'); axes[i].grid(True, alpha=0.2)
    axes[i].legend(fontsize=7)

axes[0].set_ylabel('Posterior density')
plt.suptitle(f'Bayesian updating of coin bias belief (true p={true_p})', y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

## MAP = L2 regularization (Gaussian prior on weights)

In [ ]:
rng = np.random.default_rng(1)
n = 20
X = np.column_stack([np.ones(n), rng.uniform(-2, 2, n)])
true_w = np.array([1.0, 3.0])
y = X @ true_w + rng.normal(0, 0.5, n)

def mle_solution(X, y): return np.linalg.solve(X.T @ X, X.T @ y)
def map_solution(X, y, lam): return np.linalg.solve(X.T @ X + lam*np.eye(2), X.T @ y)

w_mle = mle_solution(X, y)
lambdas = [0.1, 1.0, 5.0, 20.0]

print(f'True weights: {true_w}')
print(f'MLE:          {w_mle.round(4)}')
print()
for lam in lambdas:
    w_map = map_solution(X, y, lam)
    print(f'MAP (λ={lam:5.1f} = 1/2τ²): {w_map.round(4)}')

print('\nNote: as λ → ∞, MAP → [0, 0] (prior dominates)')
print('      as λ → 0, MAP → MLE (data dominates)')

# Plot shrinkage of w₁ and w₂ vs λ
lam_range = np.logspace(-2, 2, 100)
w0_path = [map_solution(X, y, l)[0] for l in lam_range]
w1_path = [map_solution(X, y, l)[1] for l in lam_range]

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(lam_range, w0_path, color='#6366f1', lw=2, label='w₀ (intercept)')
ax.semilogx(lam_range, w1_path, color='#f97316', lw=2, label='w₁ (slope)')
ax.axhline(true_w[0], color='#6366f1', lw=1, linestyle=':', alpha=0.7, label='True w₀')
ax.axhline(true_w[1], color='#f97316', lw=1, linestyle=':', alpha=0.7, label='True w₁')
ax.set_xlabel('λ (regularization strength)'); ax.set_ylabel('Weight value')
ax.set_title('MAP weight shrinkage as λ increases (L2 regularization / Ridge)')
ax.legend(); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()